In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import dlib
import cv2
import tensorflow as tf
import keras


In [3]:
from tensorflow.keras.applications import EfficientNetB2

def get_model():
    img_size=96
    backbone = EfficientNetB2(
        input_shape=(img_size,img_size,3),
        include_top=False,
        weights=None
    )
    mdl = tf.keras.Sequential([
        backbone,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128,activation='relu'),
        tf.keras.layers.Dense(7,activation='softmax')
    ])
    mdl.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001,
                                                    beta_1=0.9,
                                                    beta_2=0.999,
                                                    epsilon=1e-07),
                                                    loss='categorical_crossentropy',
                                                    metrics=['accuracy',
                                                    tf.keras.metrics.Precision(name='precision'),
                                                    tf.keras.metrics.Recall(name='recall')])
    return mdl
mdl=get_model()

In [4]:
#https://keras.io/api/models/model_saving_apis/keras_file_editor/
from tensorflow.keras.saving import KerasFileEditor

In [5]:
editor = KerasFileEditor("best.weights.h5")


Keras model file 'best.weights.h5'

In [6]:
#editor.summary()

In [7]:
mdl.load_weights("best.weights.h5")

c:\Users\user\OneDrive\Documents\D_I_G\ML\Deep_Learning\facial_expression_recognizer\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 608 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [8]:
mdl.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb2 (Functional)     │ (None, 3, 3, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       180,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,949,824 (30.33 MB)

 Trainable params: 7,882,249 (30.07 MB)

 Non-trainable params: 67,575 (263.97 KB)

In [9]:
import pickle
#from sklearn.preprocessing import LabelEncoder
def load_object(name:str):
    with open(f"{name}.pck",'rb') as pickle_obj:
        obj=pickle.load(pickle_obj)
    return obj
le=load_object('label_encoder')

In [10]:
def process_image(image):
    image=tf.convert_to_tensor(image)
    image=tf.image.resize(image,[96,96],method='bilinear')
    image=tf.expand_dims(image,axis=0)
    return image
def realtime_predictions(image,model,encoder_):
    prediction=model.predict(image)
    prediction=np.argmax(prediction,axis=1)
    prediction=encoder_.inverse_transform(prediction)[0]
    return prediction
def rect_to_bb(rect):
    x=rect.left()
    y=rect.top()
    w=rect.right()-x
    h=rect.bottom()-y
    return (x,y,w,h)

In [11]:
VideoCapture = cv2.VideoCapture(0)
detector = dlib.get_frontal_face_detector()

while True:

    ret , frame = VideoCapture.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    rects = detector(gray,0)

    if len(rects) >=1:
        for rect in rects:
            (x,y,w,h) = rect_to_bb(rect)
            img = gray[y-10:y+h+10,x-10:x+w+10]

            if img.shape[0] == 0 or img.shape[1] ==0:
                cv2.imshow('Frame', frame)

            else:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
                img = process_image(img)
                out = realtime_predictions(img,mdl,le)
                cv2.rectangle(frame, (x,y),(x+w,y+h),(0,255,0),2)
                z=y-15 if y-15>15 else y+15
                cv2.putText(frame,str(out),(x,z),cv2.FONT_HERSHEY_SIMPLEX,0.75,(0,255,0),2)

        cv2.imshow("Frame",frame)

    else: 
        cv2.imshow("Frame",frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

VideoCapture.release()
cv2.destroyAllWindows()



1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━